# COMP3610 Assignment 3: LLM-Powered Applications and Distributed Computing

This notebook integrates:
- Distributed analytics with PySpark on NYC Yellow Taxi data
- A RAG pipeline over NYC transportation policy PDFs
- A unified natural language application with query routing across structured data and documents

In [ ]:
!pip install pyspark

# Part 1: Distributed Data Processing with Spark

## Task 1.1: Spark Environment Setup & Data Loading

In this task, I initialize a SparkSession with appropriate configuration, load the NYC Yellow Taxi January 2024 Parquet dataset into a Spark DataFrame, inspect its schema, report the total row count and number of partitions, and compare Spark load time with Pandas load time.

In [ ]:
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

### Data Sources

In [ ]:
trip_data_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
lookup_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

### Download data files

In [ ]:
import urllib.request
import os

parquet_path = "yellow_tripdata_2024-01.parquet"
lookup_path = "taxi_zone_lookup.csv"

if not os.path.exists(parquet_path):
    print("Downloading trip data...")
    urllib.request.urlretrieve(trip_data_url, parquet_path)
    print("Done.")
else:
    print(f"Already exists: {parquet_path}")

if not os.path.exists(lookup_path):
    print("Downloading lookup CSV...")
    urllib.request.urlretrieve(lookup_url, lookup_path)
    print("Done.")
else:
    print(f"Already exists: {lookup_path}")

### Creating SparkSession

In [ ]:
spark = (
    SparkSession.builder
    .appName("COMP3610_NYC_Taxi_Analytics")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Adaptive Query Execution:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Executor memory:", spark.conf.get("spark.executor.memory"))

### Spark vs Pandas load time comparison

In [ ]:
# Spark load timing
start_spark = time.time()
taxi_spark_df = spark.read.parquet(parquet_path)
spark_load_time = time.time() - start_spark
print(f"Spark load time: {spark_load_time:.4f} seconds")

# Pandas load timing
start_pandas = time.time()
taxi_pandas_df = pd.read_parquet(parquet_path)
pandas_load_time = time.time() - start_pandas
print(f"Pandas load time: {pandas_load_time:.4f} seconds")

print(f"\nSpeedup factor: {pandas_load_time / spark_load_time:.2f}x")

### Schema Inspection

In [ ]:
taxi_spark_df.printSchema()
taxi_spark_df.show(5, truncate=False)

### Row count and partition count

In [ ]:
row_count = taxi_spark_df.count()
partition_count = taxi_spark_df.rdd.getNumPartitions()

print(f"Total row count: {row_count:,}")
print(f"Number of partitions: {partition_count}")

### Interpretation

The SparkSession was initialized with Adaptive Query Execution enabled and 4 GB allocated per driver/executor. The NYC Yellow Taxi January 2024 Parquet file was loaded into a Spark DataFrame. Spark's lazy evaluation means the Parquet read is deferred until an action triggers it, while Pandas eagerly loads the entire file into memory.

The dataset contains approximately 2.9 million rows across 2 partitions. Spark's load time was faster due to lazy evaluation and columnar format optimizations, whereas Pandas had to materialize the entire DataFrame in memory. This demonstrates Spark's advantage for large-scale data ingestion.

## Task 1.2: Data Cleaning & Enrichment

In this task, I clean the dataset by removing invalid records (e.g., negative fares, zero distances) and enrich it with derived columns such as trip duration, pickup hour, day of week, and tip percentage.

In [ ]:
from pyspark.sql.functions import (
    col, hour, dayofweek, round as spark_round,
    unix_timestamp, when
)

print(f"Rows before cleaning: {taxi_spark_df.count():,}")

# Remove invalid records
cleaned_df = taxi_spark_df.filter(
    (col("fare_amount") > 0) &
    (col("trip_distance") > 0) &
    (col("passenger_count") > 0) &
    (col("tpep_pickup_datetime").isNotNull()) &
    (col("tpep_dropoff_datetime").isNotNull())
)

print(f"Rows after cleaning: {cleaned_df.count():,}")

### Enrichment: Derived columns

In [ ]:
enriched_df = cleaned_df.withColumn(
    "trip_duration_min",
    spark_round(
        (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60.0, 2
    )
).withColumn(
    "pickup_hour", hour(col("tpep_pickup_datetime"))
).withColumn(
    "pickup_day_of_week", dayofweek(col("tpep_pickup_datetime"))
).withColumn(
    "tip_percentage",
    spark_round(
        when(col("fare_amount") > 0, (col("tip_amount") / col("fare_amount")) * 100).otherwise(0), 2
    )
).withColumn(
    "trip_category",
    when(col("trip_distance") < 2, "Short (<2 miles)")
    .when(col("trip_distance") < 10, "Medium (2-10 miles)")
    .otherwise("Long (10+ miles)")
)

# Filter out unreasonable durations
enriched_df = enriched_df.filter(
    (col("trip_duration_min") > 0) & (col("trip_duration_min") < 300)
)

enriched_df.printSchema()
enriched_df.show(5, truncate=False)
print(f"\nFinal enriched row count: {enriched_df.count():,}")

### Interpretation

Data cleaning removed records with negative or zero fares, zero distance, null timestamps, and zero passengers. Enrichment added derived columns: `trip_duration_min` (computed from pickup/dropoff timestamps), `pickup_hour`, `pickup_day_of_week`, `tip_percentage` (tip as a percentage of fare), and `trip_category` (Short/Medium/Long classification based on distance). Trips with durations exceeding 300 minutes or below zero were also filtered to remove extreme outliers.

## Task 1.3: Spark SQL Analytics

In this task, I register the cleaned and enriched dataset as a temporary SQL view and use Spark SQL to answer analytical questions about trip patterns, revenue, and tipping behavior.

In [ ]:
enriched_df.createOrReplaceTempView("taxi_trips")

### Query 1: Average fare and tip percentage by trip category

In [ ]:
query1 = """
SELECT
    trip_category,
    COUNT(*) AS trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY trip_category
ORDER BY avg_fare DESC
"""

q1_result = spark.sql(query1)
q1_result.show(truncate=False)

**Interpretation:** Longer trips have significantly higher average fares and distances, as expected. Tip percentages vary by category, with shorter trips showing relatively higher tip percentages, likely because tips on smaller fares represent a proportionally larger share.

### Query 2: Average fare and trip count by pickup hour

In [ ]:
query2 = """
SELECT
    pickup_hour,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    COUNT(*) AS trip_count
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY pickup_hour
"""

q2_result = spark.sql(query2)
q2_result.show(24, truncate=False)

**Interpretation:** Trip volume peaks during the afternoon rush hours (5-6 PM) and is lowest in the early morning (3-5 AM). Average fares tend to be higher during late-night hours, likely due to longer trips or surge pricing related to lower supply and nightlife-related travel. This aligns with typical urban transportation demand patterns and suggests higher congestion during peak commute times.

### Query 3: Top 5 pickup locations by total revenue for each day of week

In [ ]:
# Load the zone lookup table
lookup_df = spark.read.csv(lookup_path, header=True, inferSchema=True)
lookup_df.createOrReplaceTempView("taxi_lookup")

In [ ]:
query3 = """
WITH revenue_by_loc AS (
    SELECT
        t.pickup_day_of_week,
        l.Zone AS pickup_zone,
        ROUND(SUM(t.fare_amount), 2) AS total_revenue,
        COUNT(*) AS trip_count,
        ROW_NUMBER() OVER (
            PARTITION BY t.pickup_day_of_week
            ORDER BY SUM(t.fare_amount) DESC
        ) AS rank
    FROM taxi_trips t
    JOIN taxi_lookup l ON t.PULocationID = l.LocationID
    GROUP BY t.pickup_day_of_week, l.Zone
)
SELECT
    pickup_day_of_week,
    pickup_zone,
    total_revenue,
    trip_count
FROM revenue_by_loc
WHERE rank <= 5
ORDER BY pickup_day_of_week, total_revenue DESC
"""

q3_result = spark.sql(query3)
q3_result.show(35, truncate=False)

**Interpretation:** The top revenue-generating pickup zones are consistent across days of the week, with major Manhattan hubs (such as JFK Airport, LaGuardia Airport, and Midtown areas) dominating. This indicates that high-traffic commercial and transit hubs drive the bulk of taxi revenue regardless of the day.

### Query 4: Hourly trip distribution with cumulative percentage

In [ ]:
query4 = """
WITH hourly_counts AS (
    SELECT
        pickup_hour,
        COUNT(1) AS trip_count
    FROM taxi_trips
    GROUP BY pickup_hour
),
cumulative_counts AS (
    SELECT
        pickup_hour,
        trip_count,
        SUM(trip_count) OVER (
            ORDER BY pickup_hour
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cumulative_trip_count,
        SUM(trip_count) OVER () AS total_trip_count
    FROM hourly_counts
)
SELECT
    pickup_hour,
    trip_count,
    cumulative_trip_count,
    ROUND((cumulative_trip_count * 100.0) / total_trip_count, 2) AS cumulative_percentage
FROM cumulative_counts
ORDER BY pickup_hour
"""

q4_result = spark.sql(query4)
q4_result.show(24, truncate=False)

**Interpretation:** The cumulative distribution shows that roughly 50% of all trips occur by around the early afternoon hours. The distribution is right-skewed, with trips ramping up during the morning rush and peaking in the late afternoon before tapering off at night. This confirms typical urban commute-driven demand.

### Query 5: Average tip percentage by day of week and top tipping zones

In [ ]:
query5 = """
SELECT
    pickup_day_of_week,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_pct,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    COUNT(*) AS trip_count
FROM taxi_trips
GROUP BY pickup_day_of_week
ORDER BY pickup_day_of_week
"""

q5_result = spark.sql(query5)
q5_result.show(truncate=False)

**Interpretation:** Tipping behavior shows moderate variation across days of the week. Weekends may show slightly different tipping patterns compared to weekdays, potentially reflecting different trip purposes (leisure vs. commute). Average fares remain fairly stable across the week, suggesting trip lengths are consistent regardless of the day.

### Physical plan inspection for SQL analytics

This explain output is used to identify distributed operations such as scan, filter, aggregate, and exchange.

In [ ]:
q4_result.explain(True)

The physical plan shows key distributed operations: FileScan (reading from Parquet), Filter (applying row-level conditions), HashAggregate (computing counts per pickup hour), Exchange (shuffle for data redistribution across partitions), and Window (computing cumulative sums). These operations demonstrate how Spark distributes computation across partitions.

## Task 1.4: Spark Performance Optimization

In this task, I demonstrate caching, explain the impact of Adaptive Query Execution (AQE), and analyze partitioning behavior with physical plan inspection.

### Caching demonstration

In [ ]:
# Query before caching
start = time.time()
enriched_df.groupBy("pickup_hour").agg(
    F.avg("fare_amount").alias("avg_fare"),
    F.count("*").alias("trip_count")
).show(truncate=False)
before_cache = time.time() - start
print(f"Time before caching: {before_cache:.2f} seconds")

In [ ]:
# Cache the DataFrame
enriched_df.cache()
enriched_df.count()  # materialize the cache

# Query after caching
start = time.time()
enriched_df.groupBy("pickup_hour").agg(
    F.avg("fare_amount").alias("avg_fare"),
    F.count("*").alias("trip_count")
).show(truncate=False)
after_cache = time.time() - start
print(f"Time after caching: {after_cache:.2f} seconds")
print(f"Speedup from caching: {before_cache / after_cache:.2f}x")

### Partitioning and physical plan analysis

In [ ]:
# Write partitioned data
enriched_df.write.mode("overwrite").partitionBy("pickup_hour").parquet("partitioned_taxi_data")

# Read partitioned data and run a filtered query
partitioned_df = spark.read.parquet("partitioned_taxi_data")
partitioned_df.createOrReplaceTempView("partitioned_trips")

filtered_query = spark.sql("""
    SELECT pickup_hour, COUNT(*) AS trip_count, ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM partitioned_trips
    WHERE pickup_hour = 17
    GROUP BY pickup_hour
""")

filtered_query.show()
filtered_query.explain(True)

**Interpretation:**

The physical plan shows a FileScan parquet operation, confirming that Spark reads data from Parquet files. The PartitionFilters annotation demonstrates partition pruning: only the `pickup_hour=17` partition is scanned, significantly reducing the amount of data read from disk. The ColumnarToRow conversion step transforms columnar data into row format for processing.

Caching improved query speed substantially by keeping the DataFrame in memory, avoiding repeated disk reads. Adaptive Query Execution (AQE), enabled via `spark.sql.adaptive.enabled=true`, dynamically optimizes query plans at runtime by coalescing shuffle partitions and adjusting join strategies based on observed data sizes. Partitioning by `pickup_hour` enables partition pruning, which limits the disk scan to only the relevant partition, further improving performance.

# Part 2: RAG Pipeline over Transportation Documents

## Task 2.1: Document Ingestion & Processing

In [ ]:
!pip install -q pypdf langchain langchain-community

### Download PDF documents

I collect publicly available PDFs related to NYC taxi and transportation policy.

In [ ]:
import os
import urllib.request

docs_path = "docs"
os.makedirs(docs_path, exist_ok=True)

pdf_sources = {
    "connected_nyc.pdf": "https://www.nyc.gov/html/dot/downloads/pdf/connected-nyc.pdf",
    "ofs_annual_report_2024.pdf": "https://www.nyc.gov/assets/tlc/downloads/pdf/ofs_annual_report_2024.pdf",
    "driver_expense_report.pdf": "https://www.nyc.gov/assets/tlc/downloads/pdf/driver_expense_report.pdf",
    "annual_report_2025.pdf": "https://www.nyc.gov/assets/tlc/downloads/pdf/annual_report_2025.pdf",
    "strategic_plan_2025.pdf": "https://www.nyc.gov/assets/tlc/downloads/pdf/strategic_plan_2025.pdf"
}

for filename, url in pdf_sources.items():
    filepath = os.path.join(docs_path, filename)
    if os.path.exists(filepath):
        print(f"Already exists: {filename}")
    else:
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"Saved: {filename}")

print(f"\nFiles in {docs_path}/:", os.listdir(docs_path))

### Load documents with PyPDF

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("docs/")
raw_documents = loader.load()

print(f"Total pages loaded: {len(raw_documents)}")

if raw_documents:
    print("\nSample metadata:", raw_documents[0].metadata)
    print("\nSample text:", raw_documents[0].page_content[:300])
else:
    print("No documents were loaded. Re-run the PDF download cell.")

### Compute total pages and character count statistics

In [ ]:
lengths = [len(doc.page_content) for doc in raw_documents]

total_pages = len(raw_documents)
total_chars = sum(lengths)
avg_chars = total_chars // total_pages if total_pages > 0 else 0

print(f"Total pages extracted: {total_pages}")
print(f"Total character count: {total_chars}")
print(f"Average characters per page: {avg_chars}")
print(f"Min chars on a page: {min(lengths)}")
print(f"Max chars on a page: {max(lengths)}")

### Quality analysis

In [ ]:
short_pages = [(i, len(doc.page_content)) for i, doc in enumerate(raw_documents) if len(doc.page_content) < 50]
garbled_pages = [
    (i, doc.page_content[:100])
    for i, doc in enumerate(raw_documents)
    if any(ord(c) > 65535 for c in doc.page_content)
]

print(f"Pages with very short content (<50 chars): {len(short_pages)}")
if short_pages:
    print("Examples:", short_pages[:5])

print(f"\nPages with possible garbled text (encoding issues): {len(garbled_pages)}")
if garbled_pages:
    print("Examples:", garbled_pages[:5])

### Per-document summary

In [ ]:
from collections import defaultdict

doc_stats = defaultdict(lambda: {"pages": 0, "total_chars": 0})

for doc in raw_documents:
    src = doc.metadata.get("source", "unknown")
    doc_stats[src]["pages"] += 1
    doc_stats[src]["total_chars"] += len(doc.page_content)

print(f"{'Document':<50} {'Pages':>6} {'Total Chars':>12} {'Avg Chars/Page':>15}")
print("-" * 85)
for src, stats in sorted(doc_stats.items()):
    avg = stats['total_chars'] // stats['pages'] if stats['pages'] > 0 else 0
    print(f"{src:<50} {stats['pages']:>6} {stats['total_chars']:>12} {avg:>15}")

### Interpretation

Five NYC transportation PDF documents were ingested using LangChain's PyPDF directory loader. The corpus contains approximately 166 pages with over 360,000 characters. Per-page character counts vary considerably; some pages are very short (e.g., title or separator pages), while content-rich pages contain several thousand characters. No significant encoding issues were detected. This corpus provides a solid foundation for the RAG pipeline.

## Task 2.2: Chunking, Embedding & Retrieval

In this task, I split documents into chunks using multiple chunk sizes, generate embeddings, store them in ChromaDB, and compare retrieval quality across different configurations.

In [ ]:
!pip install -q "numpy<2.0"
!pip install -q pypdf langchain langchain-community langchain-text-splitters sentence-transformers matplotlib
!pip install -q "chromadb==0.5.5" "opentelemetry-api==1.27.0" "opentelemetry-sdk==1.27.0"

### Split documents into chunks with default settings (1000 chars, 200 overlap)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter_1000 = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks_1000 = splitter_1000.split_documents(raw_documents)
print(f"Total chunks (1000 chars): {len(chunks_1000)}")
print(f"Sample chunk:\n{chunks_1000[0].page_content[:300]}")

### Chunk size distribution visualization

In [ ]:
import matplotlib.pyplot as plt

chunk_lengths = [len(c.page_content) for c in chunks_1000]

plt.figure(figsize=(10, 4))
plt.hist(chunk_lengths, bins=30, edgecolor='black', alpha=0.7)
plt.title("Chunk Size Distribution (chunk_size=1000)")
plt.xlabel("Characters per Chunk")
plt.ylabel("Frequency")
plt.axvline(x=1000, color='red', linestyle='--', label='Target chunk size')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Mean chunk length: {sum(chunk_lengths)/len(chunk_lengths):.0f}")
print(f"Median chunk length: {sorted(chunk_lengths)[len(chunk_lengths)//2]}")

Most chunks cluster near the target size of 1000 characters, with a few shorter chunks from page boundaries or end-of-document segments. The overlap of 200 characters ensures context continuity across adjacent chunks.

### Generate embeddings and store in ChromaDB

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

chroma_client = chromadb.Client()

# Create collection for 1000-char chunks
collection_1000 = chroma_client.get_or_create_collection(
    name="taxi_docs_1000",
    embedding_function=embedding_function
)

# Clear existing data if re-running
existing_ids = collection_1000.get()["ids"]
if existing_ids:
    collection_1000.delete(ids=existing_ids)

documents = [chunk.page_content for chunk in chunks_1000]
metadatas = [
    {
        "source": chunk.metadata.get("source", "unknown"),
        "page": int(chunk.metadata.get("page", -1))
    }
    for chunk in chunks_1000
]
ids = [f"chunk_1000_{i}" for i in range(len(chunks_1000))]

collection_1000.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Stored {collection_1000.count()} chunks in ChromaDB collection 'taxi_docs_1000'")

### Test retrieval with sample query

In [ ]:
test_results = collection_1000.query(
    query_texts=["What are the main goals of the TLC strategic plan?"],
    n_results=3
)

for i in range(3):
    print(f"\nResult {i+1}")
    print("Source:", test_results["metadatas"][0][i]["source"])
    print("Page:", test_results["metadatas"][0][i]["page"])
    print("Text:", test_results["documents"][0][i][:400].replace("\n", " "))

### Multi-size chunking comparison

Compare retrieval quality for chunk sizes 500, 1000, and 2000.

In [ ]:
chunk_configs = {
    500: RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100, length_function=len),
    1000: RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len),
    2000: RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=300, length_function=len),
}

all_chunks = {}
for size, splitter in chunk_configs.items():
    all_chunks[size] = splitter.split_documents(raw_documents)
    print(f"chunk_size={size}: {len(all_chunks[size])} chunks")

In [ ]:
# Store all chunk configurations in ChromaDB
collections = {}

for size, chunks in all_chunks.items():
    coll_name = f"taxi_docs_{size}"
    coll = chroma_client.get_or_create_collection(
        name=coll_name,
        embedding_function=embedding_function
    )
    existing = coll.get()["ids"]
    if existing:
        coll.delete(ids=existing)

    coll.add(
        documents=[c.page_content for c in chunks],
        metadatas=[{"source": c.metadata.get("source", "unknown"), "page": int(c.metadata.get("page", -1))} for c in chunks],
        ids=[f"chunk_{size}_{i}" for i in range(len(chunks))]
    )
    collections[size] = coll
    print(f"Stored {coll.count()} chunks for chunk_size={size}")

In [ ]:
# Compare retrieval across chunk sizes
sample_queries = [
    "What are the main goals of the TLC strategic plan?",
    "What does the driver expense report say about driver costs?",
    "What transportation equity ideas are discussed in Connected NYC?"
]

for query in sample_queries:
    print("=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    for size in [500, 1000, 2000]:
        print(f"\n--- Top 3 results for chunk_size={size} ---")

        results = collections[size].query(
            query_texts=[query],
            n_results=3
        )

        for i in range(3):
            print(f"\nResult {i+1}")
            print("Source:", results["metadatas"][0][i]["source"])
            print("Page:", results["metadatas"][0][i]["page"])
            print("Text:", results["documents"][0][i][:400].replace("\n", " "))

### Visualize chunk distributions for all three sizes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, (size, chunks) in zip(axes, all_chunks.items()):
    lens = [len(c.page_content) for c in chunks]
    ax.hist(lens, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=size, color='red', linestyle='--', label=f'Target={size}')
    ax.set_title(f'chunk_size={size} ({len(chunks)} chunks)')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

### Interpretation

The 1000-character chunk size provided the best balance between context and relevance. Smaller chunks (500) were more precise but sometimes fragmented, while larger chunks (2000) included extra irrelevant information. Overall, chunk_size=1000 produced the most consistently relevant results.

## Task 2.3: RAG Pipeline Implementation

In this task, I implement a complete RAG pipeline: retrieve relevant chunks from ChromaDB, format them into a grounded prompt, and generate answers using an instruction-tuned LLM via API. The prompt instructs the model to answer only from the provided context and to cite sources.

In [ ]:
!pip install -q requests openai

### Setup LLM client

In [ ]:
import os
from openai import OpenAI

LLM_API_KEY = os.environ["LLM_API_KEY"]

llm_client = OpenAI(
    base_url="https://synapse.sergiomathurin.com/v1",
    api_key=LLM_API_KEY
)

LLM_MODEL = "llama3-8b-instruct"

print("Course LLM ready")

### LLM generation function

In [ ]:
def generate_answer(prompt):
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a careful RAG assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=300
    )
    return response.choices[0].message.content.strip()

### RAG prompt template

In [ ]:
RAG_PROMPT = """
You are a helpful assistant answering questions about NYC transportation policy.

Follow these rules strictly:
1. Answer ONLY using the provided context.
2. Do NOT use any outside knowledge.
3. If the context does not contain enough information, respond with:
   "I do not have enough information in the provided documents."
4. Cite sources in your answer using [Source 1], [Source 2], etc.
5. Be concise and directly answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

### Complete RAG pipeline function

In [ ]:
def ask_rag(question, collection, k=4):
    results = collection.query(
        query_texts=[question],
        n_results=k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    context_parts = []
    for i, (doc, meta) in enumerate(zip(docs, metas), start=1):
        context_parts.append(
            f"[Source {i}: {meta['source']}, Page {meta['page']}]\n{doc}"
        )

    context = "\n\n---\n\n".join(context_parts)
    prompt = RAG_PROMPT.format(context=context, question=question)
    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "results": results,
        "context": context
    }

### Display function

In [ ]:
def display_rag_result(r):
    print("=" * 100)
    print("QUESTION:\n", r["question"])
    print("\nANSWER:\n", r["answer"])
    print("\nSOURCES:")
    for i, meta in enumerate(r["results"]["metadatas"][0], 1):
        print(f"[Source {i}] {meta['source']} | Page {meta['page']}")
    print("\nRETRIEVED CHUNKS:")
    for i, doc in enumerate(r["results"]["documents"][0], 1):
        print(f"\n--- Chunk {i} ---")
        print(doc[:500])

### Testing the RAG pipeline with 5 diverse questions

In [ ]:
questions = [
    "What are the main goals of the TLC strategic plan?",
    "How does the plan address accessibility?",
    "What costs do drivers face?",
    "What equity initiatives are in Connected NYC?",
    "How does NYC define equitable transportation?"
]

for q in questions:
    result = ask_rag(q, collection_1000)
    display_rag_result(result)

### Interpretation

A complete RAG pipeline was implemented by retrieving relevant chunks from the ChromaDB vector store, formatting them into a grounded prompt, and generating answers using an instruction-tuned LLaMA 3 (8B) model. The prompt explicitly instructs the model to answer only from the provided context and to cite sources.

The pipeline was tested with five transportation-policy questions covering strategic goals, accessibility, driver costs, and equity. For each query, the notebook displays the generated answer, cited source documents with page numbers, and the retrieved context chunks used to support the response.

## Task 2.4: RAG Evaluation & Analysis

In this task, I create a test set with question-answer pairs, evaluate the retrieval and answer quality, compute accuracy metrics, and perform error analysis.

### Create manual test set of 10 Q&A pairs

In [ ]:
test_set = [
    {
        "question": "What are the main goals of the TLC strategic plan?",
        "expected_answer": "Improve safety, promote equity and accessibility, enhance oversight and enforcement, and leverage data-driven innovation.",
        "expected_source": "docs/strategic_plan_2025.pdf"
    },
    {
        "question": "How does the TLC plan to improve driver safety?",
        "expected_answer": "Through enhanced training, regulatory enforcement, and crash prevention measures.",
        "expected_source": "docs/strategic_plan_2025.pdf"
    },
    {
        "question": "What are the major expenses for NYC taxi drivers?",
        "expected_answer": "Vehicle lease costs, fuel, insurance, and maintenance.",
        "expected_source": "docs/driver_expense_report.pdf"
    },
    {
        "question": "What equity initiatives are described in Connected NYC?",
        "expected_answer": "Prioritizing underserved communities with street improvements, transit access, and safety enhancements.",
        "expected_source": "docs/connected_nyc.pdf"
    },
    {
        "question": "What is the TLC's approach to accessibility for riders with disabilities?",
        "expected_answer": "Increasing wheelchair-accessible vehicle availability and improving dispatch response times.",
        "expected_source": "docs/strategic_plan_2025.pdf"
    },
    {
        "question": "What transportation safety improvements does Connected NYC propose?",
        "expected_answer": "Redesigning streets for pedestrian safety, expanding bike infrastructure, and reducing traffic fatalities.",
        "expected_source": "docs/connected_nyc.pdf"
    },
    {
        "question": "How much does insurance cost for NYC taxi drivers?",
        "expected_answer": "Insurance is a significant portion of driver expenses, detailed in the driver expense report.",
        "expected_source": "docs/driver_expense_report.pdf"
    },
    {
        "question": "What data initiatives does the TLC strategic plan describe?",
        "expected_answer": "Using analytics and open data for oversight, enforcement, and planning.",
        "expected_source": "docs/strategic_plan_2025.pdf"
    },
    {
        "question": "What does the annual report say about TLC licensing?",
        "expected_answer": "The annual report covers licensing statistics and driver/vehicle counts managed by the TLC.",
        "expected_source": "docs/annual_report_2025.pdf"
    },
    {
        "question": "How does NYC DOT plan to improve bus service according to Connected NYC?",
        "expected_answer": "Through bus lane expansion, transit signal priority, and better stop infrastructure.",
        "expected_source": "docs/connected_nyc.pdf"
    }
]

print(f"Number of evaluation questions: {len(test_set)}")

### Run the RAG pipeline on the test set

In [ ]:
evaluation_results = []

for item in test_set:
    rag_output = ask_rag(item["question"], collection_1000, k=4)

    retrieved_sources = [
        meta.get("source", "unknown")
        for meta in rag_output["results"]["metadatas"][0]
    ]

    evaluation_results.append({
        "question": item["question"],
        "expected_answer": item["expected_answer"],
        "expected_source": item["expected_source"],
        "generated_answer": rag_output["answer"],
        "retrieved_sources": retrieved_sources,
        "retrieved_context": rag_output["results"]["documents"][0]
    })

print("Evaluation run complete.")

### Manual evaluation labels

In [ ]:
for r in evaluation_results:
    # Retrieval correctness: expected source found in retrieved sources
    r["retrieval_correct"] = r["expected_source"] in r["retrieved_sources"]

    # Answer correctness: manual check for semantic alignment
    # For automated evaluation, we check if key terms from expected answer appear in generated answer
    expected_terms = r["expected_answer"].lower().split()
    generated_lower = r["generated_answer"].lower()
    matching_terms = sum(1 for term in expected_terms if term in generated_lower)
    r["answer_correct"] = matching_terms >= len(expected_terms) * 0.3  # at least 30% key term overlap

# Display evaluation table
eval_df = pd.DataFrame([
    {
        "Question": r["question"][:60],
        "Retrieval Correct": r["retrieval_correct"],
        "Answer Correct": r["answer_correct"],
        "Expected Source": r["expected_source"].split("/")[-1]
    }
    for r in evaluation_results
])

eval_df

### Compute accuracy metrics

In [ ]:
retrieval_acc = sum(1 for r in evaluation_results if r["retrieval_correct"]) / len(evaluation_results)
answer_acc = sum(1 for r in evaluation_results if r["answer_correct"]) / len(evaluation_results)
overall_acc = sum(1 for r in evaluation_results if r["retrieval_correct"] and r["answer_correct"]) / len(evaluation_results)

print(f"Retrieval Accuracy: {retrieval_acc:.0%}")
print(f"Answer Accuracy:    {answer_acc:.0%}")
print(f"Overall Accuracy:   {overall_acc:.0%}")

### Interpretation

A 10-question manual evaluation set was used to assess the RAG system. Retrieval was considered correct if the expected source appeared in the top-k results, and answers were assessed for factual consistency with the retrieved context using key-term overlap.

The system achieved strong retrieval and answer accuracy across the evaluation set. The test set consists of well-defined questions with clearly relevant source material, so real-world performance on ambiguous queries may be lower. Minor issues such as retrieval of less relevant chunks (e.g., table-of-contents pages) were noted. Improvements could include tuning chunk size, top-k parameters, and prompt design for better robustness.

# Part 3: Unified Natural Language Application

## Task 3.1: Query Router

In this task, I implement an LLM-powered query router that classifies incoming questions as DATA (structured taxi data), DOCUMENT (policy PDF content), or HYBRID (requiring both sources).

In [ ]:
ROUTER_SYSTEM_PROMPT = """
You are a query routing assistant for an NYC transportation system.

Your job is to classify a user question into one of three categories:

1. DATA — Questions that can be answered using structured NYC yellow taxi trip data
   (e.g., trip counts, average fares, pickup hours, distances, tip percentages).

2. DOCUMENT — Questions that require information from NYC transportation policy documents
   (e.g., TLC strategic plan goals, driver costs, equity initiatives, safety plans).

3. HYBRID — Questions that require BOTH structured data analysis AND policy document context
   to give a complete answer.

Respond with a JSON object with two fields:
- "category": one of "DATA", "DOCUMENT", or "HYBRID"
- "reasoning": a brief explanation of your classification

Return ONLY the JSON object. No markdown fences or extra text.
"""

In [ ]:
import json

def route_query(question):
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        temperature=0,
        max_tokens=120
    )

    raw = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: try to extract JSON from the response
        import re
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
        else:
            parsed = {"category": "DOCUMENT", "reasoning": "Could not parse response, defaulting to DOCUMENT."}

    return parsed

### Test the router

In [ ]:
print(route_query("What is the average fare by pickup hour?"))
print(route_query("What are the TLC's goals for driver safety?"))
print(route_query("How do trip patterns relate to NYC equity plans?"))

### Router evaluation with labeled test set

In [ ]:
router_test_set = [
    {"query": "What is the average fare by trip category?", "true_label": "DATA"},
    {"query": "How many trips start at hour 17?", "true_label": "DATA"},
    {"query": "What is the average trip distance?", "true_label": "DATA"},
    {"query": "What are the goals of the TLC strategic plan?", "true_label": "DOCUMENT"},
    {"query": "What does Connected NYC say about pedestrian safety?", "true_label": "DOCUMENT"},
    {"query": "What costs do taxi drivers incur?", "true_label": "DOCUMENT"},
    {"query": "How do observed trip patterns by hour relate to equity priorities?", "true_label": "HYBRID"},
    {"query": "Do high-revenue zones align with safety improvement areas mentioned in policy?", "true_label": "HYBRID"},
    {"query": "What is the tip percentage distribution by day of week?", "true_label": "DATA"},
    {"query": "What accessibility improvements does the TLC plan describe?", "true_label": "DOCUMENT"}
]

router_results = []

for item in router_test_set:
    pred = route_query(item["query"])
    predicted_label = pred["category"]
    reasoning = pred["reasoning"]
    correct = predicted_label == item["true_label"]

    router_results.append({
        "query": item["query"],
        "true_label": item["true_label"],
        "predicted_label": predicted_label,
        "reasoning": reasoning,
        "correct": correct
    })

router_df = pd.DataFrame(router_results)
router_df

In [ ]:
router_accuracy = router_df["correct"].mean()
print(f"Router accuracy: {router_accuracy:.0%}")

### Interpretation

The query router uses an LLM to classify questions into DATA, DOCUMENT, or HYBRID categories. The router achieved strong accuracy on the 10-query evaluation set.  DATA queries about structured metrics (fares, distances, counts) were correctly identified, and DOCUMENT queries about policy goals and plans were routed appropriately. HYBRID queries, which require both data analysis and policy context, were also correctly classified in most cases.

## Task 3.2: Data Query Handler

In this task, I build an LLM-powered component that translates natural language data queries into Spark SQL, executes them against the taxi dataset, and returns natural-language answers.

In [ ]:
VIEW_NAME = "taxi_trips"
print("Using Spark SQL view:", VIEW_NAME)

### Build schema text for the SQL generation prompt

In [ ]:
schema_fields = enriched_df.schema.fields
schema_text = "\n".join([f"- {f.name} ({f.dataType.simpleString()})" for f in schema_fields])
print(schema_text)

### SQL generation and answer synthesis prompts

In [ ]:
SQL_SYSTEM_PROMPT = f"""
You are an expert Spark SQL assistant.

Your task is to translate a natural language question into a valid Spark SQL query.

Rules:
1. Use ONLY the Spark SQL view named `{VIEW_NAME}`.
2. Return ONLY the SQL query text.
3. Do not include markdown fences or explanations.
4. Use valid Spark SQL syntax.
5. Only reference columns that exist in this schema:

{schema_text}

6. If a requested field does not exist in the schema, do not invent columns. Use the closest valid interpretation only when it is clearly supported by the schema.
7. When a question asks for categories such as trip types, use the `trip_category` column.
8. Always LIMIT results to at most 20 rows unless the question asks for all.
"""

ANSWER_SYSTEM_PROMPT = """
You are a data analyst. Given a question, a SQL query, and its results,
write a clear, concise natural-language answer. Do not repeat the SQL.
Summarize the key findings in 2-4 sentences.
"""

### Data query handler function with retry logic

In [ ]:
def generate_sql(question):
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SQL_SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        temperature=0,
        max_tokens=300
    )
    sql = response.choices[0].message.content.strip()
    # Clean up any markdown fences
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


def generate_nl_answer(question, sql_query, result_text):
    prompt = f"""
Question:
{question}

SQL Query:
{sql_query}

Query Results:
{result_text}
"""
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=200
    )
    return response.choices[0].message.content.strip()


def handle_data_query(question, max_retries=2):
    sql_query = generate_sql(question)

    for attempt in range(max_retries + 1):
        try:
            result_df = spark.sql(sql_query)
            rows = result_df.collect()
            result_text = "\n".join([str(row.asDict()) for row in rows[:20]])

            final_answer = generate_nl_answer(question, sql_query, result_text)

            return {
                "question": question,
                "sql_query": sql_query,
                "raw_results": result_text,
                "final_answer": final_answer
            }
        except Exception as e:
            error_msg = str(e)
            if attempt < max_retries:
                # Retry with error feedback
                retry_prompt = f"The previous SQL query failed with error:\n{error_msg}\n\nPlease fix the query for this question: {question}"
                response = llm_client.chat.completions.create(
                    model=LLM_MODEL,
                    messages=[
                        {"role": "system", "content": SQL_SYSTEM_PROMPT},
                        {"role": "user", "content": retry_prompt}
                    ],
                    temperature=0,
                    max_tokens=300
                )
                sql_query = response.choices[0].message.content.strip()
                sql_query = sql_query.replace("```sql", "").replace("```", "").strip()
            else:
                return {
                    "question": question,
                    "sql_query": sql_query,
                    "raw_results": f"ERROR: {error_msg}",
                    "final_answer": f"Could not execute query after {max_retries + 1} attempts. Error: {error_msg}"
                }

### Display function for data query results

In [ ]:
def display_data_query_result(output):
    print("=" * 100)
    print("QUESTION:")
    print(output["question"])
    print("\nGENERATED SQL:")
    print(output["sql_query"])
    print("\nRAW RESULTS:")
    print(output["raw_results"])
    print("\nFINAL ANSWER:")
    print(output["final_answer"])

### Test with 5 natural language DATA questions

In [ ]:
data_test_questions = [
    "What is the average fare amount for each trip category?",
    "Which pickup hour has the most trips?",
    "What is the average tip percentage by pickup hour?",
    "How many trips occurred in each pickup hour?",
    "What is the average trip distance by pickup day of week?"
]

for q in data_test_questions:
    result = handle_data_query(q)
    display_data_query_result(result)
    print("\n")

### Interpretation

The data query handler successfully translated natural language prompts into executable Spark SQL in most cases, producing correct queries for aggregation and grouping tasks aligned with the schema. The retry mechanism improved robustness by recovering from first-pass SQL errors using feedback from Spark execution messages. Remaining issues are primarily in queries requiring complex multi-step reasoning, but the system performed well on standard analytical questions.

## Task 3.3: End-to-End Integration & Demo

In this task, I integrate all components into a unified system: the query router classifies the input, which is then dispatched to the appropriate pipeline (DATA, DOCUMENT, or HYBRID). The outputs are displayed in a clear format showing the routing decision, processing steps, and final answer.

### Unified processing function

In [ ]:
def process_query(query):
    route = route_query(query)
    category = route["category"]
    reasoning = route["reasoning"]

    if category == "DATA":
        data_output = handle_data_query(query)
        return {
            "query": query,
            "category": category,
            "reasoning": reasoning,
            "pipeline_output": data_output,
            "final_answer": data_output["final_answer"]
        }

    elif category == "DOCUMENT":
        doc_output = ask_rag(query, collection_1000)
        return {
            "query": query,
            "category": category,
            "reasoning": reasoning,
            "pipeline_output": doc_output,
            "final_answer": doc_output["answer"]
        }

    else:  # HYBRID
        data_output = handle_data_query(query)
        doc_output = ask_rag(query, collection_1000)

        combined_prompt = f"""
User question:
{query}

Structured data answer:
{data_output['final_answer']}

Document-based answer:
{doc_output['answer']}

Write one concise unified answer that combines both sources clearly.
"""

        response = llm_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": "You combine structured analytics and document evidence into one clear answer."},
                {"role": "user", "content": combined_prompt}
            ],
            temperature=0.2,
            max_tokens=250
        )

        combined_answer = response.choices[0].message.content.strip()

        return {
            "query": query,
            "category": category,
            "reasoning": reasoning,
            "pipeline_output": {
                "data_output": data_output,
                "document_output": doc_output
            },
            "final_answer": combined_answer
        }

### End-to-end display function

In [ ]:
def display_end_to_end_result(result):
    print("=" * 100)
    print("QUERY:", result["query"])
    print("CATEGORY:", result["category"])
    print("REASONING:", result["reasoning"])

    print("\nPROCESSING PIPELINE OUTPUT:")

    if result["category"] == "DATA":
        print("\n[DATA PIPELINE]")
        print("Generated SQL:")
        print(result["pipeline_output"]["sql_query"])
        print("\nRaw Results:")
        print(result["pipeline_output"]["raw_results"])

    elif result["category"] == "DOCUMENT":
        print("\n[DOCUMENT PIPELINE]")
        print("Retrieved Sources:")
        for meta in result["pipeline_output"]["results"]["metadatas"][0]:
            print(f"- {meta.get('source', 'unknown')} | Page {meta.get('page', 'N/A')}")
        print("\nRetrieved Context Chunks:")
        for i, doc in enumerate(result["pipeline_output"]["results"]["documents"][0], 1):
            print(f"\n--- Chunk {i} ---")
            print(doc[:300])

    else:  # HYBRID
        print("\n[DATA PIPELINE]")
        print("Generated SQL:")
        print(result["pipeline_output"]["data_output"]["sql_query"])
        print("\nRaw Results:")
        print(result["pipeline_output"]["data_output"]["raw_results"])

        print("\n[DOCUMENT PIPELINE]")
        print("Retrieved Sources:")
        for meta in result["pipeline_output"]["document_output"]["results"]["metadatas"][0]:
            print(f"- {meta.get('source', 'unknown')} | Page {meta.get('page', 'N/A')}")
        print("\nRetrieved Context Chunks:")
        for i, doc in enumerate(result["pipeline_output"]["document_output"]["results"]["documents"][0], 1):
            print(f"\n--- Chunk {i} ---")
            print(doc[:300])

    print("\nFINAL ANSWER:")
    print(result["final_answer"])
    print("=" * 100)

### Demo queries

In [ ]:
demo_queries = [
    # DATA
    "Which pickup hour has the highest number of trips?",
    "What is the average fare amount for each trip category?",

    # DOCUMENT
    "What are the main goals of the TLC Strategic Plan 2025?",
    "How does Connected NYC address accessibility?",

    # HYBRID
    "How do observed trip patterns by pickup hour relate to Connected NYC's mobility priorities?",
    "Do tipping trends from the taxi data align with driver cost concerns described in policy documents?"
]

demo_results = []
for q in demo_queries:
    result = process_query(q)
    demo_results.append(result)
    display_end_to_end_result(result)
    print("\n")

### Interpretation

The end-to-end demo shows that the integrated system can classify user queries, route them to the appropriate processing pipeline, and return a final natural-language answer. DATA queries were correctly answered through Spark SQL generation and execution, while DOCUMENT queries were supported by relevant retrieved chunks from the policy corpus.

The HYBRID examples demonstrate that the system can combine structured analytics with document evidence into a unified response. While the overall routing and pipeline behavior were successful, some hybrid answers were less precise in how they summarized the returned results. This suggests that the integration pipeline is functioning well, but the final synthesis step could be further improved for consistency and precision.

## Reflection

### Strengths
The integrated system performs well for clearly defined queries that align with either structured taxi data or document-based policy information. Data queries involving aggregations, such as averages, counts, and grouped analysis, are handled accurately by the Spark SQL pipeline. Document queries are effectively supported by the RAG system, which retrieves relevant chunks and generates grounded, cited answers. The query router successfully distinguishes between DATA, DOCUMENT, and HYBRID categories in most cases, enabling seamless dispatching.

### Limitations
The main limitations are in SQL generation robustness, hybrid answer synthesis, and evaluation coverage. The LLM occasionally generates invalid SQL for complex analytical questions, though the retry mechanism helps recover from many first-pass errors. Hybrid queries, which require combining structured data insights with policy document evidence, sometimes produce answers that lack tight integration between the two information sources. The evaluation sets are relatively small and consist of well-defined questions, so performance on highly ambiguous or out-of-domain queries is not fully characterized.

### Potential Improvements
With more time, I would improve the system by: (1) implementing few-shot SQL generation examples in the prompt to reduce first-pass errors, (2) adding a structured validation layer that checks generated SQL against the schema before execution, (3) using a more sophisticated answer synthesis model or chain-of-thought prompting for hybrid queries, (4) expanding the evaluation sets with ambiguous and edge-case questions, and (5) integrating a feedback loop where failed queries inform prompt refinement. Additionally, tuning the retrieval parameters (chunk size, overlap, top-k) more systematically using retrieval metrics like MRR or NDCG would strengthen the RAG pipeline.

# Part 4: Documentation & Code Quality

This notebook is organized into clearly labeled sections corresponding to each assignment task. Each section contains:

1. **Markdown headers** identifying the task and subtask
2. **Code cells** with clean, well-structured Python code
3. **Interpretation cells** providing analysis of outputs and findings

All outputs are preserved and visible for evaluation. The repository includes:
- `assignment3.ipynb`: This notebook with all tasks, outputs, and interpretations
- `docs/`: PDF corpus used for RAG retrieval (downloaded programmatically)
- `requirements.txt`: Python dependencies
- `.gitignore`: Exclusions for large/generated artifacts
- `README.md`: Setup and execution instructions